# Gale dust devil layer: join all inputs into one per-pixel CSV

Builds `gale_dust_devil_features_100m.csv`: one row per 100 m pixel on the THEMIS template grid, with every input the likelihood demo uses.
The resampling matches `../dust_devil/dust_devil_likelihood_demo.ipynb` cell for cell (area-average for DEM/slope, sin/cos average for aspect, bilinear for albedo), so the numbers agree with the demo.

**Outputs** (in `processed_data/`)
- `gale_dust_devil_features_100m.csv`: pixel_id, row, col, lon, lat, elev_m, slope_deg, aspect_deg, ti_themis_tiu, albedo_my24/25/26, albedo_mean
- `gale_dust_devil_meta.json`: constants that are the same for every pixel (sol, Ls, tau, rover pixel, template grid)

No try/except: a missing file or a grid mismatch stops the run.
Needs: numpy, pandas, rasterio.

In [1]:
from pathlib import Path
import json
import numpy as np, pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling

PROC = Path("../processed_data")
BINNED_CSV = PROC / "rems_binned_all_sols.csv"
TEMPLATE_TIF = PROC / "gale_ti_night_themis100m_native.tif"          # output grid (100 m)
DEM_TIF, SLOPE_TIF, ASPECT_TIF = PROC/"gale_dem_20m.tif", PROC/"gale_slope_20m.tif", PROC/"gale_aspect_20m.tif"
ALB_TIFS = {y: PROC/f"gale_albedo_my{y}_native.tif" for y in (24, 25, 26)}

DEMO_SOL = 800
TAU = 0.5                 # PLACEHOLDER dust opacity, same as the demo
OUT_CSV  = PROC / "gale_dust_devil_features_100m.csv"
OUT_META = PROC / "gale_dust_devil_meta.json"
MIN_TI_COVERAGE = 0.99    # the brief says THEMIS covers the whole box; fail if it does not

In [2]:
with rasterio.open(TEMPLATE_TIF) as t:
    T_TF, T_CRS, T_SHAPE = t.transform, t.crs, (t.height, t.width)
    ti = t.read(1).astype("float32")
    if t.nodata is not None: ti[ti == t.nodata] = np.nan
assert T_CRS.is_geographic, "template should be lon/lat"
print("template", T_SHAPE, "pixel (deg):", T_TF.a, T_TF.e)
print("template bounds lon %.4f..%.4f  lat %.4f..%.4f" % (T_TF.c, T_TF.c + T_TF.a*T_SHAPE[1], T_TF.f + T_TF.e*T_SHAPE[0], T_TF.f))
ti_cov = np.isfinite(ti).mean()
assert ti_cov >= MIN_TI_COVERAGE, f"THEMIS covers only {ti_cov:.1%} of the template"
print(f"THEMIS thermal inertia coverage: {ti_cov:.2%}")

template (1436, 1477) pixel (deg): 0.001687051876845213 -0.0016870993139129456
template bounds lon 136.7491..139.2408  lat -6.5713..-4.1486
THEMIS thermal inertia coverage: 100.00%


C:\Users\aypen\AppData\Local\Temp\ipykernel_20972\3147389954.py:3: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  ti = t.read(1).astype("float32")


In [3]:
def to_template(path, how=Resampling.average, band_fn=None):
    with rasterio.open(path) as s:
        a = s.read(1).astype("float32")
        if s.nodata is not None: a[a == s.nodata] = np.nan
        if band_fn is not None: a = band_fn(a)
        out = np.full(T_SHAPE, np.nan, "float32")
        reproject(a, out, src_transform=s.transform, src_crs=s.crs, dst_transform=T_TF, dst_crs=T_CRS,
                  resampling=how, src_nodata=np.nan, dst_nodata=np.nan)
    return out

dem   = to_template(DEM_TIF)
slope = to_template(SLOPE_TIF)
asp_s = to_template(ASPECT_TIF, band_fn=lambda a: np.sin(np.radians(a)))   # average aspect as sin/cos, never raw degrees
asp_c = to_template(ASPECT_TIF, band_fn=lambda a: np.cos(np.radians(a)))
aspect = np.degrees(np.arctan2(asp_s, asp_c)) % 360
alb = {y: to_template(p, Resampling.bilinear) for y, p in ALB_TIFS.items()}
alb_mean = np.nanmean(np.stack(list(alb.values())), axis=0)

layers = dict(elev_m=dem, slope_deg=slope, aspect_deg=aspect, ti_themis_tiu=ti,
              albedo_my24=alb[24], albedo_my25=alb[25], albedo_my26=alb[26], albedo_mean=alb_mean)
for n, a in layers.items():
    print(f"{n:14s} min {np.nanmin(a):9.3f}  max {np.nanmax(a):9.3f}  NaN {100*np.mean(~np.isfinite(a)):5.1f}%")

C:\Users\aypen\AppData\Local\Temp\ipykernel_20972\1667873775.py:3: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  a = s.read(1).astype("float32")
C:\Users\aypen\AppData\Local\Temp\ipykernel_20972\1667873775.py:3: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  a = s.read(1).astype("float32")
C:\Users\aypen\AppData\Local\Temp\ipykernel_20972\1667873775.py:3: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  a = s.read(1).astype("float32")
C:\Users\aypen\AppData\Local\Temp\ipykernel_20972\1667873775.py:3: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
A

elev_m         min -4715.405  max  1115.874  NaN   0.0%
slope_deg      min     0.000  max    82.097  NaN   0.0%
aspect_deg     min     0.000  max   360.000  NaN   0.0%
ti_themis_tiu  min   102.231  max   818.643  NaN   0.0%
albedo_my24    min     0.103  max     0.273  NaN   0.0%
albedo_my25    min     0.097  max     0.263  NaN   0.0%
albedo_my26    min     0.100  max     0.278  NaN   0.0%
albedo_mean    min     0.105  max     0.265  NaN   0.0%


In [4]:
rows, cols = np.meshgrid(np.arange(T_SHAPE[0]), np.arange(T_SHAPE[1]), indexing="ij")
lat = T_TF.f + (rows + 0.5) * T_TF.e
lon = T_TF.c + (cols + 0.5) * T_TF.a
df = pd.DataFrame({"pixel_id": (rows * T_SHAPE[1] + cols).ravel().astype("int32"),
                   "row": rows.ravel().astype("int32"), "col": cols.ravel().astype("int32"),
                   "lon": lon.ravel(), "lat": lat.ravel(),
                   **{n: a.ravel() for n, a in layers.items()}})
n_all = len(df)
df = df.dropna(subset=list(layers)).reset_index(drop=True)
print(f"kept {len(df):,} of {n_all:,} pixels ({len(df)/n_all:.1%}); dropped {n_all-len(df):,} with a missing layer")
assert len(df) > 0.9 * n_all, "more than 10% of pixels lack a layer: check that the 20 m rasters cover the template"
df = df.round({"lon": 6, "lat": 6, "elev_m": 2, "slope_deg": 3, "aspect_deg": 2, "ti_themis_tiu": 2,
               "albedo_my24": 5, "albedo_my25": 5, "albedo_my26": 5, "albedo_mean": 5})   # %.5g would cut lon/lat to 0.01 deg
df.to_csv(OUT_CSV, index=False)
print("wrote", OUT_CSV, f"({OUT_CSV.stat().st_size/1e6:.0f} MB)")
df.head()

kept 2,120,824 of 2,120,972 pixels (100.0%); dropped 148 with a missing layer
wrote processed_data\gale_dust_devil_features_100m.csv (207 MB)


,pixel_id,row,col,lon,lat,elev_m,slope_deg,aspect_deg,ti_themis_tiu,albedo_my24,albedo_my25,albedo_my26,albedo_mean
0,0,0,0,136.749895,-4.149421,-2288.429932,7.773,306.429993,255.320007,0.23349,0.22679,0.22774,0.22934
1,1,0,1,136.751582,-4.149421,-2282.290039,5.173,325.079987,286.329987,0.23348,0.22693,0.22772,0.22937
2,2,0,2,136.753269,-4.149421,-2275.360107,8.967,295.649994,305.250000,0.23346,0.22707,0.22770,0.22941
3,3,0,3,136.754956,-4.149421,-2252.590088,16.114,278.049988,297.239990,0.23345,0.22721,0.22768,0.22945
4,4,0,4,136.756643,-4.149421,-2225.169922,13.235,268.779999,316.109985,0.23344,0.22735,0.22765,0.22948


In [5]:
# ---- constants that are the same for every pixel (copied from the demo so Ls matches) ----
SOL_DAYS = 1.0274912517
LANDING_JD_TT = 2456145.7 + 69.184/86400
def ls_deg(sol):
    jd = LANDING_JD_TT + np.asarray(sol, float) * SOL_DAYS
    dj = jd - 2451545.0
    M = np.radians(19.3871 + 0.52402073 * dj)
    alpha = 270.3863 + 0.52403840 * dj
    pbs = (0.0071*np.cos(np.radians(0.985626*dj/2.2353+49.409)) + 0.0057*np.cos(np.radians(0.985626*dj/2.7543+168.173))
           + 0.0039*np.cos(np.radians(0.985626*dj/1.1177+191.837)) + 0.0037*np.cos(np.radians(0.985626*dj/15.7866+21.736))
           + 0.0021*np.cos(np.radians(0.985626*dj/2.1354+15.704)) + 0.0020*np.cos(np.radians(0.985626*dj/2.4694+95.528))
           + 0.0018*np.cos(np.radians(0.985626*dj/32.8493+49.095)))
    eoc = (10.691 + 3.0e-7*dj)*np.sin(M) + 0.623*np.sin(2*M) + 0.050*np.sin(3*M) + 0.005*np.sin(4*M) + 0.0005*np.sin(5*M) + pbs
    return (alpha + eoc) % 360

rems = pd.read_csv(BINNED_CSV)
day = rems[rems.sol == DEMO_SOL].sort_values("bin").reset_index(drop=True)
assert len(day) > 0, f"sol {DEMO_SOL} not in {BINNED_CSV.name}"
assert day.latitude_deg.nunique() == 1 and day.longitude_deg_e.nunique() == 1, "rover moved during the demo sol"
c, r = ~T_TF * (day.longitude_deg_e[0], day.latitude_deg[0]); r, c = int(r), int(c)   # affine returns (col, row)
assert 0 <= r < T_SHAPE[0] and 0 <= c < T_SHAPE[1], "rover pixel is outside the template"

meta = dict(demo_sol=DEMO_SOL, ls_deg=float(ls_deg(DEMO_SOL)), tau=TAU, tau_is_placeholder=True,
            rover_lat=float(day.latitude_deg[0]), rover_lon=float(day.longitude_deg_e[0]), rover_elev_m=float(day.elevation_m[0]),
            rover_row=r, rover_col=c, rover_pixel_id=int(r*T_SHAPE[1]+c),
            n_rems_bins=int(len(day)), template_shape=list(T_SHAPE), template_transform=list(T_TF)[:6])
OUT_META.write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

# sanity: DEM at the rover pixel vs the elevation REMS reports for the rover
dem_rover = float(dem[r, c])
print(f"DEM at rover pixel {dem_rover:.1f} m vs REMS elevation {meta['rover_elev_m']:.1f} m (diff {dem_rover-meta['rover_elev_m']:+.1f} m)")
print("rover-pixel row in the CSV:", int((df.pixel_id == meta["rover_pixel_id"]).sum()), "(expect 1)")

{
  "demo_sol": 800,
  "ls_deg": 228.18267085792286,
  "tau": 0.5,
  "tau_is_placeholder": true,
  "rover_lat": -4.662574,
  "rover_lon": 137.377518,
  "rover_elev_m": -4461.2,
  "rover_row": 304,
  "rover_col": 372,
  "rover_pixel_id": 449380,
  "n_rems_bins": 60,
  "template_shape": [
    1436,
    1477
  ],
  "template_transform": [
    0.001687051876845213,
    0.0,
    136.7490510333193,
    0.0,
    -0.0016870993139129456,
    -4.148577212911933
  ]
}
DEM at rover pixel -4437.0 m vs REMS elevation -4461.2 m (diff +24.2 m)
rover-pixel row in the CSV: 1 (expect 1)
